# E4-B — Audit de traçabilité RDF → jeu supervisé → représentations finales

**But scientifique :** mesurer les correspondances identifiables entre les observations extraites du graphe RDF et les valeurs d'entrée/cible des modèles, sans confondre *correspondance temporelle* et *provenance prouvée*.

**Sources** : `04_sparql/03_ml_extraction_long_traceable.csv`, `06_common_experiment/01_common_supervised_dataset.csv`, archive/dossier `07D_final_alignment`, fichiers `05B_data_quality` (optionnels). **Horizon primaire : 15 min**.

**Précautions** : les observations RDF sont horodatées à la seconde et les échantillons sont placés sur une grille de 15 min. L'association dans ce notebook utilise `floor('15min')` comme **hypothèse de rapprochement à vérifier** contre le code original de synchronisation. Une concordance de valeur ne prouve **pas** qu'une observation a alimenté une valeur transformée. Les séries peuvent avoir subi agrégation et interpolation. Aucune mesure de gain de RMSE ou de gain d'efficacité du RDF n'est effectuée.

**Sorties** : vérifications des partitions et des valeurs, audit de toutes les 15 caractéristiques d'entrée (5×3) et des 3 variables de la cible possible (la cible réelle de température +15 min est évaluée), résumé de couverture et manifeste des limites.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import json, hashlib, time
import numpy as np
import pandas as pd
from IPython.display import display

# --- Sur ta machine : place le dossier outputs au même niveau que ce notebook,
# ou indique son chemin absolu ci-dessous.
OUTPUTS_SOURCE = Path('outputs')  # e.g., Path('outputs')

# Archive 07D facultative si outputs/07D_final_alignment existe déjà.
ALIGNMENT_ZIP = Path('07D_final_alignment(1).zip')

# Répertoire de CSV déposés directement à côté du notebook (facultatif).
EXTRA_FILES_DIR = Path('.')

RESULT_DIR = Path('E4_B_lineage_results')
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# False : conserver uniquement les résumés et les 92 316 statuts par échantillon.
# True : exporter les ~1,66 million de correspondances de caractéristiques.
EXPORT_FEATURE_LEVEL = False

GRID_MINUTES = 15
ATOL = 1e-4  # tolérance numérique (en unité native de chaque variable)
RTOL = 1e-6

def locate_file(rel, aliases=()):
    candidates = [OUTPUTS_SOURCE / rel, Path(rel), EXTRA_FILES_DIR / Path(rel).name]
    candidates.extend(EXTRA_FILES_DIR / a for a in aliases)
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError('Fichier absent : ' + rel + '\nChemins examinés : ' + ', '.join(map(str,candidates)))

def read_alignment(name, **kw):
    direct = OUTPUTS_SOURCE / '07D_final_alignment' / name
    if direct.is_file():
        return pd.read_csv(direct, **kw)
    alt = Path('07D_final_alignment') / name
    if alt.is_file():
        return pd.read_csv(alt, **kw)
    zip_options = [ALIGNMENT_ZIP, OUTPUTS_SOURCE / '07D_final_alignment.zip', Path('07D_final_alignment(1).zip')]
    for zpath in zip_options:
        if zpath.is_file():
            with ZipFile(zpath) as z:
                hits = [p for p in z.namelist() if p.endswith('/'+name) or p == name]
                if hits:
                    with z.open(hits[0]) as f:
                        return pd.read_csv(f, **kw)
    raise FileNotFoundError(f'07D_final_alignment/{name} introuvable ; configure OUTPUTS_SOURCE ou ALIGNMENT_ZIP.')

def utc(s):
    return pd.to_datetime(s, utc=True, errors='raise')

print('Python/Pandas :', pd.__version__)
print('Source :', OUTPUTS_SOURCE.resolve())
print('Sortie :', RESULT_DIR.resolve())

## 1. Charger et identifier les sources

In [ ]:
rdf_path = locate_file('04_sparql/03_ml_extraction_long_traceable.csv', ('03_ml_extraction_long_traceable(2).csv','03_ml_extraction_long_traceable(1).csv'))
sup_path = locate_file('06_common_experiment/01_common_supervised_dataset.csv', ('01_common_supervised_dataset(2).csv',))
rdf = pd.read_csv(rdf_path, usecols=['uuid','timestamp','variable','value','unit','point','observation'])
sup = pd.read_csv(sup_path)
final_keys = read_alignment('01_final_common_sample_keys.csv')
tab = read_alignment('02_tabular_core_15_unscaled.csv')
meta_seq = {s: read_alignment(f'metadata_sequence_{s}.csv') for s in ('train','validation','test')}
meta_graph = {s: read_alignment(f'metadata_graph_{s}.csv') for s in ('train','validation','test')}
for df in (rdf,sup,final_keys,tab):
    df['uuid'] = df['uuid'].astype(str)
for df in (rdf,sup,final_keys,tab):
    df['timestamp'] = utc(df['timestamp'])
for df in (sup,tab):
    df['target_timestamp'] = utc(df['target_timestamp'])
for s in meta_seq:
    meta_seq[s]['timestamp']=utc(meta_seq[s]['timestamp'])
    meta_seq[s]['target_timestamp']=utc(meta_seq[s]['target_timestamp'])
    meta_graph[s]['target_timestamp']=utc(meta_graph[s]['target_timestamp'])

required={'uuid','timestamp','split','target_timestamp','temperature','relative_humidity','co2','target_temperature_t_plus_15','forecast_horizon_minutes'}
missing=required-set(sup.columns)
assert not missing, f'Colonnes manquantes dans supervised : {missing}'
assert sup['forecast_horizon_minutes'].dropna().eq(15).all(), 'ERREUR: jeu supervisé différent du protocole principal à 15 min.'
assert tab['target_timestamp'].sub(tab['timestamp']).dt.total_seconds().eq(900).all(), 'Cible autre que +15 min détectée.'

print('Lignes RDF :',len(rdf),'— UUID :',rdf.uuid.nunique(),'— variables :',rdf.variable.value_counts().to_dict())
print('Lignes supervisées :',len(sup),'— horizons :',sup.forecast_horizon_minutes.unique())
print('Clés finales :',len(final_keys),'— tableau :',len(tab))
print('RDF unités :',rdf.groupby('variable')['unit'].unique().to_dict())

## 2. Cohérence : dataset supervisé → ensemble final → fenêtres séquentielles/graphiques

In [ ]:
KEY=['uuid','timestamp','split']
assert not sup.duplicated(KEY).any(), 'Clés supervisées dupliquées'
assert not final_keys.duplicated(KEY).any(), 'Clés finales dupliquées'
assert not tab.duplicated(KEY).any(), 'Clés tabulaires finales dupliquées'

j=final_keys.merge(sup[KEY + ['temperature','target_temperature_t_plus_15','target_timestamp']],on=KEY,how='left',validate='one_to_one',indicator=True)
assert j['_merge'].eq('both').all(), 'Certaines clés finales sont absentes du supervisé'
t=tab.merge(j.drop(columns=['_merge']),on=KEY,how='left',validate='one_to_one',suffixes=('_tab','_sup'))
assert len(t)==len(tab)

val_current=np.isclose(t['temperature_t'],t['temperature'],rtol=RTOL,atol=ATOL,equal_nan=True)
val_target=np.isclose(t['target_temperature_t_plus_15_tab'],t['target_temperature_t_plus_15_sup'],rtol=RTOL,atol=ATOL,equal_nan=True)
time_target=(t['target_timestamp_tab']==t['target_timestamp_sup'])
seq_keys=pd.concat([df[KEY] for df in meta_seq.values()],ignore_index=True)
seq_check=final_keys.merge(seq_keys.drop_duplicates(KEY),on=KEY,how='outer',indicator=True)

rows=[]
for s in ('train','validation','test'):
    n_sup=(sup.split==s).sum();n_fin=(final_keys.split==s).sum()
    rows.append({'split':s,'supervised':n_sup,'aligned':n_fin,'removed':n_sup-n_fin,
                 'sequence_samples':len(meta_seq[s]), 'graph_snapshots':len(meta_graph[s]),
                 'graph_node_predictions_expected':len(meta_graph[s])*7})
counts=pd.DataFrame(rows)
counts.loc[len(counts)]={'split':'TOTAL',**{c:int(counts[c].sum()) for c in counts.columns if c!='split'}}
display(counts)
counts.to_csv(RESULT_DIR/'01_stage_counts.csv',index=False)
checks={'supervised_unique_keys':bool(not sup.duplicated(KEY).any()),
        'all_final_keys_in_supervised':bool(j['_merge'].eq('both').all()),
        'current_temperature_equal':int(val_current.sum()),
        'target_temperature_equal':int(val_target.sum()),
        'target_timestamp_equal':int(time_target.sum()),
        'total_final':len(tab),
        'final_vs_sequence_keys_equal':bool(seq_check['_merge'].eq('both').all()),
        'seven_node_snapshots':bool(all(len(meta_graph[s])*7==len(meta_seq[s]) for s in meta_seq))}
print('Contrôles :',json.dumps(checks,indent=2))
assert val_current.all() and val_target.all() and time_target.all(), 'Valeur cible ou T courante non concordante'
assert checks['final_vs_sequence_keys_equal'] and checks['seven_node_snapshots']
(RESULT_DIR/'02_alignment_checks.json').write_text(json.dumps(checks,indent=2),encoding='utf8')

## 3. Préparer un index de candidats RDF par UUID, variable et grille 15 min

**Important :** une observation brute `2025-...T11:15:21Z` est rangée ici dans le créneau `11:15`, par convention **floor**. La convention réelle de synchronisation est à vérifier avant d'affirmer un lignage exact. Les duplications dans le même créneau restent visibles via `n_raw`.

In [ ]:
rdf['grid_time']=rdf['timestamp'].dt.floor(f'{GRID_MINUTES}min')
rdf['value']=pd.to_numeric(rdf['value'],errors='coerce')
assert rdf['value'].notna().all(),'Certaines valeurs RDF ne sont pas numériques'
raw_group = rdf.groupby(['uuid','variable','grid_time'],sort=False,observed=True).agg(
    n_raw=('observation','size'),
    raw_min=('value','min'), raw_max=('value','max'), raw_mean=('value','mean'),
    raw_first=('value','first'),
    unique_observation=('observation','first'),
    n_unique_obs=('observation','nunique'),
    n_unique_points=('point','nunique')
).reset_index()
raw_group['collision']=raw_group.n_raw.gt(1)
raw_group.to_csv(RESULT_DIR/'03_rdf_grid_collisions.csv',index=False)
print('Groupes UUID-variable-créneau :',len(raw_group))
print('Créneaux avec plusieurs observations :',raw_group.collision.sum())
print('Observations dans ces créneaux :',raw_group.loc[raw_group.collision,'n_raw'].sum())
print('Rappel: une collision n’est pas nécessairement un doublon RDF ; plusieurs mesures par quart d’heure sont possibles.')

## 4. Auditer chaque variable et chaque pas temporel des entrées

Pour chaque échantillon final : **15 caractéristiques d'entrée** (`t−60, −45, −30, −15, t` × température/RH/CO₂) **et une cible température à `t+15`**. Une valeur correspondante est classée en :

- `single_candidate_value_match`: une observation brute candidate dans le créneau et valeur égale à la valeur finale (tolérance numérique), **sans preuve du processus exact** ;
- `multiple_candidates_value_in_range`: plusieurs observations candidates, valeur finale dans leur intervalle ; l'opération précise n'est pas identifiée ;
- `candidate_value_mismatch`: présence de candidat(s), valeur hors de l'intervalle brut (interpolation/transformation possibles) ;
- `no_candidate_in_grid`: aucune observation candidate dans le créneau ;

Aucune classe n'est étiquetée *provenance exacte*. Ce contrôle ne reconstruit pas les règles de calcul et **ne certifie pas** l'attribution d'un identifiant d'observation à une valeur transformée.

In [ ]:
feature_specs=[]
for lag in [60,45,30,15,0]:
    suffix='t' if lag==0 else f't_minus_{lag}'
    for var,colprefix in [('temperature','temperature'),('relative_humidity','relative_humidity'),('co2','co2')]:
        feature_specs.append((f'{colprefix}_{suffix}',var,-lag))
feature_specs.append(('target_temperature_t_plus_15','temperature',15))
assert all(c in tab.columns for c,_,_ in feature_specs)

feature_counts=[]
row_stat=pd.DataFrame({'uuid':tab['uuid'],'timestamp':tab['timestamp'],'split':tab['split']})
row_stat['n_features']=len(feature_specs)
for c in ('single_candidate_value_match','multiple_candidates_value_in_range','candidate_value_mismatch','no_candidate_in_grid'):
    row_stat[c]=0
feature_exports=[]
start=time.time()
for feature,var,offset in feature_specs:
    query=pd.DataFrame({'uuid':tab['uuid'], 'grid_time':tab['timestamp']+pd.to_timedelta(offset,unit='m')})
    query['variable']=var
    hit=query.merge(raw_group,on=['uuid','variable','grid_time'],how='left',sort=False,validate='many_to_one')
    assert len(hit)==len(tab)
    numeric=pd.to_numeric(tab[feature],errors='coerce').to_numpy(dtype=float)
    count=hit['n_raw'].fillna(0).to_numpy(dtype=int)
    low=hit['raw_min'].to_numpy(dtype=float)
    high=hit['raw_max'].to_numpy(dtype=float)
    close_one=np.isclose(numeric,hit['raw_first'].to_numpy(dtype=float),rtol=RTOL,atol=ATOL)
    tolerance=ATOL+RTOL*np.maximum(np.abs(numeric),np.maximum(np.abs(low),np.abs(high)))
    in_range=(numeric>=low-tolerance)&(numeric<=high+tolerance)
    status=np.select([count==0,(count==1)&close_one,(count>1)&in_range],
      ['no_candidate_in_grid','single_candidate_value_match','multiple_candidates_value_in_range'],
      default='candidate_value_mismatch')
    for category in ('single_candidate_value_match','multiple_candidates_value_in_range','candidate_value_mismatch','no_candidate_in_grid'):
        row_stat[category] += (status==category).astype(int)
    vc=pd.Series(status).value_counts()
    feature_counts.append({'feature':feature,'variable':var,'offset_minutes':offset,'n_samples':len(tab),
      **{cat:int(vc.get(cat,0)) for cat in ('single_candidate_value_match','multiple_candidates_value_in_range','candidate_value_mismatch','no_candidate_in_grid')}})
    if EXPORT_FEATURE_LEVEL:
        feature_exports.append(pd.DataFrame({'uuid':tab.uuid,'timestamp':tab.timestamp,'split':tab.split,
            'feature':feature,'raw_candidates':count,'status':status,'final_value':numeric,
            'raw_min':low,'raw_max':high}))
    print(f'{feature:38s} | absent={vc.get("no_candidate_in_grid",0):5d} | unique concordant={vc.get("single_candidate_value_match",0):6d} | multiple={vc.get("multiple_candidates_value_in_range",0):6d} | écart={vc.get("candidate_value_mismatch",0):6d}')

feature_summary=pd.DataFrame(feature_counts)
feature_summary.to_csv(RESULT_DIR/'04_feature_level_summary.csv',index=False)
row_stat['all_features_have_candidate'] = row_stat['no_candidate_in_grid'].eq(0)
row_stat['all_features_unique_and_equal'] = row_stat['single_candidate_value_match'].eq(len(feature_specs))
row_stat.to_csv(RESULT_DIR/'05_final_sample_status.csv',index=False)
if EXPORT_FEATURE_LEVEL:
    pd.concat(feature_exports,ignore_index=True).to_csv(RESULT_DIR/'06_optional_feature_level_audit.csv',index=False)
print('Temps (s):',round(time.time()-start,1))
print('Table 16 variables par échantillon, N=',len(tab))
display(feature_summary)

## 5. Produire les indicateurs sans confondre couverture et provenance

In [ ]:
overall={'n_final_samples':len(tab),'n_checked_values':len(tab)*len(feature_specs),
    'n_input_features_per_sample':15,'n_target_features_per_sample':1,
    'all_16_feature_grid_candidates_samples':int(row_stat['all_features_have_candidate'].sum()),
    'all_16_features_unique_value_match_samples':int(row_stat['all_features_unique_and_equal'].sum())}
for col in ('single_candidate_value_match','multiple_candidates_value_in_range','candidate_value_mismatch','no_candidate_in_grid'):
    overall[col]=int(row_stat[col].sum())
    overall[col+'_fraction']=float(row_stat[col].sum()/overall['n_checked_values'])
print(json.dumps(overall,indent=2))
(RESULT_DIR/'07_e4b_aggregate_summary.json').write_text(json.dumps(overall,indent=2),encoding='utf8')
by_split=row_stat.groupby('split').agg(samples=('uuid','size'),all_16_candidates=('all_features_have_candidate','sum'),
 all_16_unique_and_equal=('all_features_unique_and_equal','sum'),
 no_raw_candidate_values=('no_candidate_in_grid','sum'),
 candidate_mismatch_values=('candidate_value_mismatch','sum'),
 ambiguous_multicandidate_values=('multiple_candidates_value_in_range','sum')).reset_index()
by_split.to_csv(RESULT_DIR/'08_summary_by_split.csv',index=False)
display(by_split)
# Comparaison secondaire limitée aux seules 3 variables au temps t
current_features=feature_summary[feature_summary.offset_minutes.eq(0)]
print('Couverture de trois variables au temps t (agrégée par variable, pas intersection des lignes):')
display(current_features[['feature','single_candidate_value_match','multiple_candidates_value_in_range','candidate_value_mismatch','no_candidate_in_grid']])

## 6. Vérification documentaire et limites (à conserver dans toute réponse au reviewer)

L'enregistrement des transformations exactes exigerait les tables/algorithmes de la synchronisation, l'affectation de plusieurs observations dans un créneau et les marqueurs d'interpolation. **On ne doit pas assimiler** `single_candidate_value_match` à une preuve de provenance causale. La chaîne **supervisé → final**, en revanche, est testée directement par identité de clés et de valeurs. Le fait que les données proviennent d'un graphe ne démontre pas un avantage sur une chaîne CSV non sémantique.

Le dossier `05B_data_quality` documente des contrôles de qualité, mais les fichiers de synthèse ne sont pas des journaux de provenance par mesure.

In [ ]:
quality_manifest=None
try:
    qpath=locate_file('05B_data_quality/11_cleaning_decision_manifest.csv',('11_cleaning_decision_manifest(1).csv',))
    quality_manifest=pd.read_csv(qpath)
    display(quality_manifest)
except FileNotFoundError:
    print('Manifeste de nettoyage facultatif absent : audit poursuivi sans cet élément.')

manifest={
    'experiment':'E4-B RDF-to-supervised-to-final lineage audit',
    'source_rdf':str(rdf_path),'source_supervised':str(sup_path),
    'alignment':'07D_final_alignment/02_tabular_core_15_unscaled.csv',
    'source_hash_sha256':{str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in (rdf_path,sup_path)},
    'method':'Raw RDF timestamps grouped by floor(15 min), with UUID and variable; compare 16 sample values per final row to raw candidates in same grid slot.',
    'matched_supervised_final':checks,
    'summary':overall,
    'limit_1':'The true synchronization convention and row-level transformation lineage are not independently verified.',
    'limit_2':'Same-grid candidate and numerical equality are not proof of which original RDF observation generated a value.',
    'limit_3':'Multiple raw observations and interpolated values cannot be disambiguated without processing logs or original transformation code.',
    'limit_4':'No empirical advantage of semantic vs. nonsemantic preprocessing or forecasting is tested.',
    'input_model':'5x3 node-local sensor history plus one +15 min temperature target, horizon 15 minutes',
    'status':'descriptive evidence; not fully verified exact RDF lineage',
    'pandas':pd.__version__
}
(RESULT_DIR/'09_protocol_provenance_limitations.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf8')
print('Résultats enregistrés dans :',RESULT_DIR.resolve())
print('IMPORTANT : aucune nouvelle affirmation scientifique ne doit être insérée avant interprétation des sorties et confirmation de la règle de synchronisation.')